# Step 6: External RefMet Error Taxonomy

This notebook completes Step 6 from `results/plans.md`: build an error taxonomy for the corrected external lab `metabolite_name -> refmet_name` evaluation.

It is self-contained and intended for Colab. It does not re-encode embeddings or rerun FAISS. It reads the final prediction files already generated in `results/external_lab_refmet_model_comparison_reproducible/`.

Outputs:

- `results/error_taxonomy.csv`
- `results/error_taxonomy_summary.md`
- `results/external_lab_refmet_model_comparison_reproducible/step6_error_taxonomy.csv`
- `results/external_lab_refmet_model_comparison_reproducible/step6_manual_review_sheet.csv`
- `results/external_lab_refmet_model_comparison_reproducible/step6_representative_examples.csv`
- `results/external_lab_refmet_model_comparison_reproducible/plots/step6_error_taxonomy_counts.png`

## 1. Imports and Optional Drive Mount

If running in Colab, mount Drive first. If the repo is already available in the runtime, the mount step is harmless to skip.

In [ ]:
from pathlib import Path
import json
import math
import re
import unicodedata
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')

SEED = 17
rng = np.random.default_rng(SEED)
pd.set_option('display.max_colwidth', 120)
print('Imports ready. IN_COLAB =', IN_COLAB)

## 2. Configure Paths

Set `PROJECT_ROOT` to the repository root. In the default Colab setup used throughout this project, that is usually `/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer`.

In [ ]:
candidate_roots = [
    Path.cwd().resolve(),
    Path.cwd().resolve().parent,
    Path('/content/drive/MyDrive/ChemSynonymizer/chem_synonymizer'),
    Path('/content/chem_synonymizer'),
]

PROJECT_ROOT = None
for root in candidate_roots:
    if (root / 'results' / 'external_lab_refmet_model_comparison_reproducible').exists():
        PROJECT_ROOT = root
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the repository root. Set PROJECT_ROOT manually in this cell.')

RESULTS_DIR = PROJECT_ROOT / 'results'
RUN_DIR = RESULTS_DIR / 'external_lab_refmet_model_comparison_reproducible'
PLOTS_DIR = RUN_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

PREDICTIONS_PATH = RUN_DIR / 'v4_final_test_predictions.csv'
TOP_HITS_PATH = RUN_DIR / 'v4_final_test_top_hits.csv'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RUN_DIR =', RUN_DIR)

## 3. Load Final Test Predictions

The taxonomy is based on final held-out test predictions from the corrected external RefMet task. A false positive means the reranked top-1 RefMet label is not the exact gold `refmet_name`. Identity-aware correctness is kept as a separate diagnostic.

In [ ]:
if not PREDICTIONS_PATH.exists():
    raise FileNotFoundError(PREDICTIONS_PATH)
if not TOP_HITS_PATH.exists():
    raise FileNotFoundError(TOP_HITS_PATH)

pred = pd.read_csv(PREDICTIONS_PATH)
top_hits = pd.read_csv(TOP_HITS_PATH)

pred['top1_exact_correct'] = pred['top1_exact_correct'].astype(bool)
pred['top1_identity_correct'] = pred['top1_identity_correct'].astype(bool)
pred['confidence_margin'] = pd.to_numeric(pred['confidence_margin'], errors='coerce')
pred['reranker_exact_rank'] = pd.to_numeric(pred['reranker_exact_rank'], errors='coerce')
pred['reranker_identity_rank'] = pd.to_numeric(pred['reranker_identity_rank'], errors='coerce')

print('Predictions:', pred.shape)
print('Top hits:', top_hits.shape)
display(pred.head(3))

## 4. Taxonomy Helper Functions

These rules are intentionally transparent. They are not meant to replace human review; they create a first-pass taxonomy and a manual-review sheet. A paper should describe these as heuristic labels unless manually verified.

In [ ]:
GREEK_MAP = {
    'α': 'alpha', 'β': 'beta', 'γ': 'gamma', 'δ': 'delta',
    'ε': 'epsilon', 'κ': 'kappa', 'λ': 'lambda', 'μ': 'mu', 'ω': 'omega',
}

def normalize_text(value):
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKC', str(value))
    for src, dst in GREEK_MAP.items():
        text = text.replace(src, dst)
    text = text.lower().strip()
    return re.sub(r'\s+', ' ', text)

def compact_key(text):
    return re.sub(r'[^a-z0-9]+', '', normalize_text(text))

def token_set(text):
    return {tok for tok in re.split(r'[^a-z0-9]+', normalize_text(text)) if tok}

def char_ngrams(text, n=3):
    s = compact_key(text)
    if not s:
        return set()
    if len(s) <= n:
        return {s}
    return {s[i:i+n] for i in range(len(s) - n + 1)}

def jaccard(a, b):
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))

def seq_ratio(a, b):
    a = normalize_text(a)
    b = normalize_text(b)
    if len(a) > 120 or len(b) > 120:
        return jaccard(char_ngrams(a), char_ngrams(b))
    return SequenceMatcher(None, a, b).ratio()

def split_cids(value):
    if pd.isna(value):
        return set()
    out = set()
    for part in str(value).split('|'):
        part = part.strip()
        if not part or part.lower() == 'nan':
            continue
        if part.endswith('.0') and part[:-2].isdigit():
            part = part[:-2]
        out.add(part)
    return out

def stereo_tokens(text):
    s = normalize_text(text)
    return set(re.findall(r'\b(?:cis|trans|alpha|beta)\b|\b[dl]-|\([+-]\)|\b[0-9]+[rs]\b|\b[0-9]+[ez]\b|\([0-9rs,\-]+\)', s))

LIPID_RE = re.compile(r'\b(?:pc|pe|pg|pi|ps|pa|tg|dg|mg|sm|cer|dhcer|hexcer|lpc|lpe|fa|car|ce)\s*[\( ]?\s*\d{1,2}\s*:\s*\d{1,2}', re.I)
SALT_FORM_RE = re.compile(r'\b(?:hydrochloride|chloride|sodium|potassium|calcium|magnesium|ammonium|hydrate|anhydrous|salt|sulfate|sulphate|phosphate|acetate|formate|adduct|oxide|ester|methyl ester|ethyl ester)\b', re.I)
ABBREV_RE = re.compile(r'^[A-Z0-9\-\s]{2,12}$')

def has_lipid(text):
    return bool(LIPID_RE.search(str(text)))

def has_salt_form(text):
    return bool(SALT_FORM_RE.search(str(text)))

def is_abbrev_like(text):
    s = str(text).strip()
    compact = compact_key(s)
    toks = token_set(s)
    if ABBREV_RE.match(s) and len(compact) <= 12:
        return True
    if len(toks) <= 2 and len(compact) <= 10 and any(ch.isdigit() for ch in compact):
        return True
    if len(toks) == 1 and len(compact) <= 8:
        return True
    return False

def category_for_row(row):
    query = row.get('metabolite_name', '')
    gold = row.get('refmet_name', '')
    pred_name = row.get('top1_refmet_name', '')
    qn = row.get('metabolite_normalized', normalize_text(query))
    gn = row.get('refmet_normalized', normalize_text(gold))
    pn = row.get('top1_refmet_normalized', normalize_text(pred_name))

    q_tokens = token_set(qn)
    g_tokens = token_set(gn)
    p_tokens = token_set(pn)
    categories = []
    evidence = []

    exact_ok = bool(row.get('top1_exact_correct', False))
    identity_ok = bool(row.get('top1_identity_correct', False))
    exact_rank = row.get('reranker_exact_rank', np.nan)
    identity_rank = row.get('reranker_identity_rank', np.nan)
    gold_cids = split_cids(row.get('gold_refmet_cids', ''))
    pred_cids = split_cids(row.get('top1_refmet_cids', ''))

    if exact_ok:
        categories.append('success_exact_top1')
    elif identity_ok:
        categories.append('identity_match_not_exact_label')

    if (not exact_ok) and pd.notna(exact_rank) and 1 < float(exact_rank) <= 10:
        categories.append('reranking_error_gold_in_top10')
        evidence.append(f'gold exact rank={int(exact_rank)}')
    elif (not identity_ok) and pd.notna(identity_rank) and 1 < float(identity_rank) <= 10:
        categories.append('reranking_error_identity_in_top10')
        evidence.append(f'gold identity rank={int(identity_rank)}')
    elif (not exact_ok) and (pd.isna(exact_rank) or float(exact_rank) > 10):
        categories.append('candidate_generation_miss_or_low_rank')

    if (not gold_cids) or (not pred_cids):
        categories.append('reference_label_limitation')
        evidence.append('missing CID evidence for gold or predicted label')

    if compact_key(gn) == compact_key(pn) and gn != pn:
        categories.append('formatting_only_mismatch')
        evidence.append('same compact key but different normalized strings')

    if stereo_tokens(qn) or stereo_tokens(gn) or stereo_tokens(pn):
        if stereo_tokens(gn) != stereo_tokens(pn):
            categories.append('stereochemistry_confusion')
            evidence.append(f'gold stereo={sorted(stereo_tokens(gn))}; predicted stereo={sorted(stereo_tokens(pn))}')

    if has_salt_form(qn) or has_salt_form(gn) or has_salt_form(pn):
        if has_salt_form(gn) != has_salt_form(pn) or has_salt_form(qn):
            categories.append('salt_adduct_form_confusion')

    if has_lipid(qn) or has_lipid(gn) or has_lipid(pn):
        categories.append('lipid_shorthand_ambiguity')

    if is_abbrev_like(query) or is_abbrev_like(gold) or is_abbrev_like(pred_name):
        categories.append('abbreviation_ambiguity')

    if q_tokens and g_tokens and (q_tokens < g_tokens or g_tokens < q_tokens):
        categories.append('token_omission')
        evidence.append('query/gold token set is a strict subset')

    lexical_sim = seq_ratio(gn, pn)
    char_sim = jaccard(char_ngrams(gn), char_ngrams(pn))
    tok_sim = jaccard(g_tokens, p_tokens)
    if (not identity_ok) and (lexical_sim >= 0.70 or char_sim >= 0.55 or tok_sim >= 0.50):
        categories.append('near_isomer_or_near_neighbor_confusion')
        evidence.append(f'seq={lexical_sim:.2f}; char3={char_sim:.2f}; token={tok_sim:.2f}')

    if not categories:
        categories.append('unclassified_needs_manual_review')

    primary = next((c for c in categories if not c.startswith('success')), categories[0])
    return pd.Series({
        'primary_error_category': primary,
        'all_error_categories': '|'.join(dict.fromkeys(categories)),
        'taxonomy_evidence': '; '.join(evidence),
        'gold_pred_sequence_ratio': lexical_sim,
        'gold_pred_char3_jaccard': char_sim,
        'gold_pred_token_jaccard': tok_sim,
    })

print('Taxonomy helpers ready.')

## 5. Apply Taxonomy to All Test Predictions

This creates row-level category labels for the full test prediction file. Multi-label categories are stored in `all_error_categories`; `primary_error_category` is the first paper-facing category used for counts.

In [ ]:
taxonomy_features = pred.apply(category_for_row, axis=1)
taxonomy = pd.concat([pred.copy(), taxonomy_features], axis=1)

taxonomy['case_group'] = np.select(
    [
        taxonomy['top1_exact_correct'],
        (~taxonomy['top1_exact_correct']) & taxonomy['top1_identity_correct'],
        (~taxonomy['top1_exact_correct']) & taxonomy['reranker_exact_rank'].between(2, 10, inclusive='both'),
        (~taxonomy['top1_identity_correct']) & taxonomy['reranker_identity_rank'].between(2, 10, inclusive='both'),
    ],
    [
        'exact_top1_success',
        'identity_only_top1_success',
        'gold_exact_in_top10_not_top1',
        'gold_identity_in_top10_not_top1',
    ],
    default='top1_failure_or_gold_not_in_top10',
)

taxonomy['needs_manual_review'] = ~taxonomy['top1_exact_correct']

root_taxonomy_path = RESULTS_DIR / 'error_taxonomy.csv'
run_taxonomy_path = RUN_DIR / 'step6_error_taxonomy.csv'
taxonomy.to_csv(root_taxonomy_path, index=False)
taxonomy.to_csv(run_taxonomy_path, index=False)

print('Saved:', root_taxonomy_path)
print('Saved:', run_taxonomy_path)
display(taxonomy[['query_id', 'case_group', 'primary_error_category', 'metabolite_name', 'refmet_name', 'top1_refmet_name', 'reranker_exact_rank', 'confidence_margin']].head(10))

## 6. Create Manual Review Samples

The plan asks for 100 false positives, 50 true positives with high confidence, and 50 cases where the correct answer appears in top 10 but not rank 1. This cell creates those samples and includes blank columns for human review.

In [ ]:
def sample_or_all(df, n, sort_col=None, ascending=False, random=False):
    if df.empty:
        return df.copy()
    if sort_col is not None and sort_col in df.columns:
        df = df.sort_values(sort_col, ascending=ascending)
    if len(df) <= n:
        return df.copy()
    if random:
        return df.sample(n=n, random_state=SEED)
    return df.head(n).copy()

false_positive = taxonomy[~taxonomy['top1_exact_correct']].copy()
false_positive['sample_type'] = 'false_positive_top1_exact_wrong'
false_positive = sample_or_all(false_positive, 100, sort_col='confidence_margin', ascending=False)

true_positive = taxonomy[taxonomy['top1_exact_correct']].copy()
true_positive['sample_type'] = 'high_confidence_true_positive'
true_positive = sample_or_all(true_positive, 50, sort_col='confidence_margin', ascending=False)

gold_in_top10 = taxonomy[(~taxonomy['top1_exact_correct']) & taxonomy['reranker_exact_rank'].between(2, 10, inclusive='both')].copy()
gold_in_top10['sample_type'] = 'gold_in_top10_not_rank1'
gold_in_top10 = sample_or_all(gold_in_top10, 50, sort_col='reranker_exact_rank', ascending=True)

manual_review = pd.concat([false_positive, true_positive, gold_in_top10], ignore_index=True)
manual_review['manual_primary_category'] = ''
manual_review['manual_secondary_categories'] = ''
manual_review['manual_notes'] = ''

review_cols = [
    'sample_type', 'query_id', 'source_row_id', 'metabolite_name', 'refmet_name', 'top1_refmet_name',
    'metabolite_normalized', 'refmet_normalized', 'top1_refmet_normalized',
    'top1_exact_correct', 'top1_identity_correct', 'top1_identity_reason',
    'reranker_exact_rank', 'reranker_identity_rank', 'confidence_margin',
    'gold_refmet_cids', 'top1_refmet_cids',
    'primary_error_category', 'all_error_categories', 'taxonomy_evidence',
    'manual_primary_category', 'manual_secondary_categories', 'manual_notes',
]
manual_review = manual_review[[c for c in review_cols if c in manual_review.columns]]

manual_review_path = RUN_DIR / 'step6_manual_review_sheet.csv'
manual_review.to_csv(manual_review_path, index=False)
print('Saved:', manual_review_path, manual_review.shape)
display(manual_review.head(10))

## 7. Representative Examples and Summary Tables

This cell creates category counts, representative examples, and plots. These artifacts can be used directly in the paper discussion and limitations sections.

In [ ]:
category_counts = (
    taxonomy[~taxonomy['top1_exact_correct']]
    .groupby('primary_error_category')
    .agg(
        n_cases=('query_id', 'count'),
        mean_confidence_margin=('confidence_margin', 'mean'),
        mean_gold_pred_sequence_ratio=('gold_pred_sequence_ratio', 'mean'),
        identity_correct_rate=('top1_identity_correct', 'mean'),
    )
    .reset_index()
    .sort_values('n_cases', ascending=False)
)
category_counts['pct_of_top1_exact_failures'] = category_counts['n_cases'] / max(1, int((~taxonomy['top1_exact_correct']).sum()))

case_group_counts = taxonomy['case_group'].value_counts().rename_axis('case_group').reset_index(name='n_cases')
case_group_counts['pct_of_test'] = case_group_counts['n_cases'] / len(taxonomy)

category_counts_path = RUN_DIR / 'step6_error_taxonomy_counts.csv'
case_group_counts_path = RUN_DIR / 'step6_case_group_counts.csv'
category_counts.to_csv(category_counts_path, index=False)
case_group_counts.to_csv(case_group_counts_path, index=False)

rep_examples = []
for category, group in taxonomy[~taxonomy['top1_exact_correct']].groupby('primary_error_category'):
    group = group.sort_values(['confidence_margin', 'gold_pred_sequence_ratio'], ascending=[False, False]).head(5)
    rep_examples.append(group)
representative = pd.concat(rep_examples, ignore_index=True) if rep_examples else pd.DataFrame()
rep_cols = [
    'primary_error_category', 'query_id', 'metabolite_name', 'refmet_name', 'top1_refmet_name',
    'reranker_exact_rank', 'reranker_identity_rank', 'top1_identity_correct', 'confidence_margin',
    'gold_refmet_cids', 'top1_refmet_cids', 'all_error_categories', 'taxonomy_evidence',
]
representative = representative[[c for c in rep_cols if c in representative.columns]]
representative_path = RUN_DIR / 'step6_representative_examples.csv'
representative.to_csv(representative_path, index=False)

plt.figure(figsize=(12, 6))
plot_df = category_counts.sort_values('n_cases', ascending=True)
plt.barh(plot_df['primary_error_category'], plot_df['n_cases'])
plt.xlabel('Number of top-1 exact failures')
plt.title('Step 6 error taxonomy counts')
plt.tight_layout()
plot_path = PLOTS_DIR / 'step6_error_taxonomy_counts.png'
plt.savefig(plot_path, dpi=180)
plt.show()

print('Saved:', category_counts_path)
print('Saved:', case_group_counts_path)
print('Saved:', representative_path)
print('Saved:', plot_path)
display(category_counts)
display(case_group_counts)
display(representative.head(20))

## 8. Write Markdown Summary

This writes a concise Step 6 summary that can be copied into the paper's Results or Discussion section.

In [ ]:
n_test = len(taxonomy)
n_exact_success = int(taxonomy['top1_exact_correct'].sum())
n_identity_success = int(taxonomy['top1_identity_correct'].sum())
n_exact_fail = n_test - n_exact_success
n_gold_top10_not_top1 = int(((~taxonomy['top1_exact_correct']) & taxonomy['reranker_exact_rank'].between(2, 10, inclusive='both')).sum())
n_candidate_miss = int(((~taxonomy['top1_exact_correct']) & (taxonomy['reranker_exact_rank'].isna() | (taxonomy['reranker_exact_rank'] > 10))).sum())

lines = []
lines.append('# Step 6 Error Taxonomy Summary')
lines.append('')
lines.append('This summary analyzes final held-out test predictions from the corrected external laboratory RefMet normalization task.')
lines.append('')
lines.append('## Overall Counts')
lines.append('')
lines.append(f'- Test queries analyzed: {n_test:,}')
lines.append(f'- Exact top-1 successes: {n_exact_success:,} ({n_exact_success / n_test:.2%})')
lines.append(f'- Identity-aware top-1 successes: {n_identity_success:,} ({n_identity_success / n_test:.2%})')
lines.append(f'- Exact top-1 failures: {n_exact_fail:,} ({n_exact_fail / n_test:.2%})')
lines.append(f'- Gold exact target in top 10 but not rank 1: {n_gold_top10_not_top1:,} ({n_gold_top10_not_top1 / n_test:.2%})')
lines.append(f'- Gold exact target absent from top 10 or unranked: {n_candidate_miss:,} ({n_candidate_miss / n_test:.2%})')
lines.append('')
lines.append('## Primary Error Categories')
lines.append('')
lines.append('| Category | Cases | Percent of exact failures | Identity-correct rate | Mean confidence margin |')
lines.append('| --- | ---: | ---: | ---: | ---: |')
for _, row in category_counts.iterrows():
    lines.append(
        f"| {row['primary_error_category']} | {int(row['n_cases']):,} | {row['pct_of_top1_exact_failures']:.2%} | "
        f"{row['identity_correct_rate']:.2%} | {row['mean_confidence_margin']:.4f} |"
    )
lines.append('')
lines.append('## Interpretation')
lines.append('')
lines.append('The taxonomy separates ranking failures from candidate-generation failures. Cases where the gold target appears in the top 10 but not at rank 1 are especially important because they indicate that the semantic retrieval stage found the right neighborhood, but the final reranking decision selected a competing label. Cases where the gold target is absent from the top 10 are more likely to require better candidate generation, more complete target dictionaries, synonym expansion, or stronger normalization rules.')
lines.append('')
lines.append('The heuristic labels should be treated as a first-pass taxonomy. The manual review sheet contains sampled false positives, high-confidence true positives, and gold-in-top-10 reranking errors for human validation before publication.')
lines.append('')
lines.append('## Generated Artifacts')
lines.append('')
lines.append(f'- Full row-level taxonomy: `{root_taxonomy_path.relative_to(PROJECT_ROOT)}`')
lines.append(f'- Run-local taxonomy: `{run_taxonomy_path.relative_to(PROJECT_ROOT)}`')
lines.append(f'- Manual review sheet: `{manual_review_path.relative_to(PROJECT_ROOT)}`')
lines.append(f'- Category counts: `{category_counts_path.relative_to(PROJECT_ROOT)}`')
lines.append(f'- Representative examples: `{representative_path.relative_to(PROJECT_ROOT)}`')
lines.append(f'- Category plot: `{plot_path.relative_to(PROJECT_ROOT)}`')

summary_text = '\n'.join(lines) + '\n'
root_summary_path = RESULTS_DIR / 'error_taxonomy_summary.md'
run_summary_path = RUN_DIR / 'step6_error_taxonomy_summary.md'
root_summary_path.write_text(summary_text, encoding='utf-8')
run_summary_path.write_text(summary_text, encoding='utf-8')

print('Saved:', root_summary_path)
print('Saved:', run_summary_path)
print(summary_text[:2500])